# Card or cash? Predicting how NYC taxi riders pay
**CS4207 Coding Challenge · Group 23 · University of Limerick**

Henry Sampers · Fuwhad Akinwale · David Bradshaw · Rayan Blaq · Donal Reynolds

**Problem.** For each yellow-taxi trip, predict whether it was paid by **card or cash**
(TLC `payment_type` 1 or 2) from what is known about the trip: distance, time, fare,
fees, rate code and the pickup/drop-off zones.

**Data.** NYC TLC yellow-taxi trip records, one Parquet file per month, stored on Rockhill in
`/home/cs4207/data` (33 files, January 2024 to September 2026). The split is by time:

* **train:** January 2024 – September 2025
* **validation window:** October – December 2025 (used to choose the model)
* **test:** every month of 2026 (January – September), never seen until the very end

**How the notebook is organised** (each part maps to a rubric criterion):

| Part | What it does | Parallel pattern |
|---|---|---|
| 0 | Configuration, environment, hardware | – |
| 1 | Inspect file footers, clean each month | **data parallelism** (one shard per month) |
| 2 | Combine partial zone counts; check against the sequential result | merge of partial results |
| 3 | Build feature groups | **task parallelism #1: feature extraction** |
| 4 | Train and compare 6 model settings | **task parallelism #2: hyperparameter search** |
| 5 | Sequential vs parallel benchmark, speed-up chart, Amdahl's law | – |
| 6 | Non-determinism: two sources shown, fixed and re-checked | – |
| 7 | Final test on April 2026 | – |
| 8 | Summary (every number used on the poster) | – |

Run with *Restart & Run All*, or from a terminal:
`jupyter nbconvert --to notebook --execute payment_method.ipynb --output payment_method.ipynb`

## Part 0 · Configuration (all settings in one place)

In [ ]:
import os
# Pin maths libraries to one thread per process BEFORE numpy is imported: workers then do not
# fight over cores, and results do not depend on how many threads the library picks.
for var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[var] = "1"

# ---- settings ---------------------------------------------------------------------------
DATA_DIR      = os.environ.get("TAXI_DATA_DIR", "/home/cs4207/data")
PATTERN       = os.environ.get("TAXI_PATTERN", "yellow_tripdata_{month}.parquet")
import pandas as pd   # (after the thread settings above)

def months_between(first, last):
    return [str(p) for p in pd.period_range(first, last, freq="M")]

def months_from_env(name, default):
    value = os.environ.get(name)
    return value.split(",") if value else default

TRAIN_MONTHS  = months_from_env("TAXI_TRAIN", months_between("2024-01", "2025-09"))
VAL_MONTHS    = months_from_env("TAXI_VAL",   months_between("2025-10", "2025-12"))  # validation window
TEST_MONTHS   = months_from_env("TAXI_TEST",  months_between("2026-01", "2026-09"))  # used once, at the end
SAMPLE        = float(os.environ.get("TAXI_SAMPLE", "0.05"))  # share of each train/val month used
TEST_SAMPLE   = 1.0        # 2026 is tested in full
SEED          = 4207       # base seed; shard i uses SEED + i
SMOOTHING     = 200        # zone cash rates: zones with few trips are pulled to the overall rate
MAX_WORKERS   = 8          # Rockhill has 64 cores but is shared, so we stay modest
BENCH_SHARDS  = 8          # cleaning benchmark uses the first 8 monthly files
CLEAN_WORKER_COUNTS   = [1, 2, 4, 8]   # 8 shards
FEATURE_WORKER_COUNTS = [1, 2, 4]      # 4 feature groups
SEARCH_WORKER_COUNTS  = [1, 2, 3, 6]   # 6 model settings
REPEATS       = 3          # timed repeats per worker count (after one warm-up run per stage)
BENCH_ROWS    = 200_000    # training rows used in the search benchmark (keeps it short)
OUT, OUTPUTS  = "results", "outputs"   # big intermediate files / small files for the poster

GRID = [   # every setting is one independent task in Part 4
    ("logreg_C1",        "logreg", {"C": 1.0}),
    ("logreg_C1_bal",    "logreg", {"C": 1.0, "class_weight": "balanced"}),
    ("logreg_C0.01_bal", "logreg", {"C": 0.01, "class_weight": "balanced"}),
    ("hgb_100x31",       "hgb", {"max_iter": 100, "max_leaf_nodes": 31, "learning_rate": 0.1}),
    ("hgb_200x63",       "hgb", {"max_iter": 200, "max_leaf_nodes": 63, "learning_rate": 0.1}),
    ("hgb_200x63_bal",   "hgb", {"max_iter": 200, "max_leaf_nodes": 63, "learning_rate": 0.1,
                                 "class_weight": "balanced"}),
]
# -------------------------------------------------------------------------------------------

import json, platform, subprocess, sys, time
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score)
import payment_workers as pw

NOTEBOOK_START = time.perf_counter()
for d in (OUT, f"{OUT}/clean", f"{OUT}/clean_seq", f"{OUT}/features", f"{OUT}/prepared",
          f"{OUT}/models", f"{OUT}/demo", OUTPUTS):
    os.makedirs(d, exist_ok=True)
pd.set_option("display.width", 160)

### Environment and hardware

In [ ]:
def total_ram_gb():
    try:
        with open("/proc/meminfo") as fh:
            return round(int(fh.readline().split()[1]) / 1024**2, 1)
    except OSError:
        return None

env = {
    "python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
    "pyarrow": pyarrow.__version__, "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "host": platform.node(), "os": platform.platform(),
    "cpu_count": os.cpu_count(),
    "usable_cores": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count(),
    "ram_gb": total_ram_gb(),
    "load_average_1_5_15min": list(os.getloadavg()) if hasattr(os, "getloadavg") else None,
    "started": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open(f"{OUTPUTS}/environment.json", "w") as fh:
    json.dump(env, fh, indent=2)
pd.Series(env, name="value").to_frame()

## Part 1 · Loading and cleaning (data parallelism)

### 1a. Look at the file footers first
A Parquet footer holds the row count, row groups and schema, so we can check every file
**without loading any data**.

In [ ]:
jobs   = ([(m, "train", SAMPLE) for m in sorted(TRAIN_MONTHS)] + [(m, "val", SAMPLE) for m in sorted(VAL_MONTHS)]
          + [(m, "test", TEST_SAMPLE) for m in sorted(TEST_MONTHS)])
months = [m for m, _, _ in jobs]
roles  = [r for _, r, _ in jobs]
fracs  = [f for _, _, f in jobs]
paths  = [os.path.join(DATA_DIR, PATTERN.format(month=m)) for m in months]
missing = [p for p in paths if not os.path.exists(p)]
assert not missing, f"taxi files not found: {missing}"

footer = pd.DataFrame([{
    "file": os.path.basename(p), "role": r,
    "rows": pq.ParquetFile(p).metadata.num_rows,
    "row_groups": pq.ParquetFile(p).metadata.num_row_groups,
    "columns": pq.ParquetFile(p).metadata.num_columns,
    "size_MB": round(os.path.getsize(p) / 1e6, 1)} for p, r in zip(paths, roles)])
print(f"{len(jobs)} monthly files: {len(TRAIN_MONTHS)} train, {len(VAL_MONTHS)} validation, "
      f"{len(TEST_MONTHS)} test; {footer.rows.sum():,} rows in total")
footer

### 1b. Cleaning rules and why
Each monthly file is one **shard**; the same function `pw.clean_month` cleans every shard.
Only the 16 columns we need are read (column pruning). Steps, in order:

| Step | Rule | Why |
|---|---|---|
| Flex Fare | drop `payment_type` 0 | Flex Fare trips have no card/cash label, and their passenger count, rate code and fee columns are null. They cannot be used for training or testing, so we drop them on purpose rather than filling them in. |
| Other payment types | keep only 1 (card) and 2 (cash) | No charge, dispute, unknown and voided trips are not the question. |
| Spill-over | pickup must be inside the file's month | Each file contains a few trips from the month before or after (late uploads, wrong clocks). Without this a December or May trip could leak into the wrong split. |
| Fare | 0 < fare < $500 | Negative fares are refunds or corrections; $500+ is above any normal metered trip in NYC. |
| Distance | 0 < distance < 100 miles | Zero-distance trips are cancelled or meter errors; some trips report 200,000+ miles. 100 miles covers any trip to the airports and nearby counties. |
| Duration | 1 to 180 minutes | Dropoff before pickup or under a minute is a clock error; over 3 hours is a meter left running. |
| Nulls | passenger count → 1, rate code → 1 (standard) | After the steps above, few nulls remain. We use the usual value and report how many were filled. |
| Categories | rate code → one 0/1 column per code; zones → smoothed cash rate | Rate code has 6 values (one-hot). There are ~260 zones, so each is replaced by its cash rate from training rows only (Part 2). |
| Leakage | `tip_amount`, `total_amount` are never read | Card tips are recorded automatically but cash tips almost never are, so a tip gives the answer away. |
| Split | train = Jan 2024 – Sep 2025; validation = Oct – Dec 2025; test = all of 2026 | Time-based: the model is chosen on later months than it was trained on, and tested on a later year. |

There are about 60 million card or cash trips in 2024–2025. To keep the shared server responsive,
training and validation months use a fixed random sample (`SAMPLE`, 5% of every month, seeded per
shard), which is still about 3 million trips. The 2026 test months are used in full.

In [ ]:
clean_paths = [f"{OUT}/clean/{m}_{r}.parquet" for m, r in zip(months, roles)]
seeds = [SEED + i for i in range(len(jobs))]       # each shard always gets the same seed

def clean_args(shards=None, out_paths=clean_paths):
    """Argument lists for pw.clean_month; shards = which monthly files (default: all)."""
    idx = range(len(jobs)) if shards is None else shards
    pick = lambda xs: [xs[i] for i in idx]
    return [pick(paths), pick(months), pick(out_paths), pick(roles), pick(fracs), pick(seeds)]

def run_tasks(fn, arg_lists, workers):
    """workers == 1: a plain loop in this process (the sequential version).
    workers > 1: a process pool. pool.map returns results in INPUT order, not in the
    order the workers finish, so the combined result never depends on timing."""
    if workers == 1:
        return list(map(fn, *arg_lists))
    with ProcessPoolExecutor(max_workers=workers) as pool:
        return list(pool.map(fn, *arg_lists))

def timed(fn, *args):
    t0 = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - t0

stats, clean_s = timed(run_tasks, pw.clean_month, clean_args(), min(MAX_WORKERS, len(jobs)))
print(f"cleaned {len(jobs)} shards in {clean_s:.1f}s with {min(MAX_WORKERS, len(jobs))} workers")
shards = pd.DataFrame([{k: s[k] for k in ("month", "role", "rows_clean", "rows_kept", "cash_share",
                                           "seconds", "pid", "md5")} for s in stats])
shards.to_csv(f"{OUTPUTS}/shards.csv", index=False)
shards

### 1c. Rows before and after each cleaning step

In [ ]:
per_month = pd.DataFrame({s["month"]: {**dict(s["steps"]), "rows after cleaning": s["rows_clean"],
                                       "rows used (after sampling)": s["rows_kept"]} for s in stats}).T
per_month.insert(0, "role", roles)
per_month.to_csv(f"{OUTPUTS}/cleaning_per_month.csv")
cleaning = per_month.groupby("role", sort=False).sum().T.astype("int64")   # steps x (train, val, test)
cleaning["total"] = cleaning.sum(axis=1)
cleaning["% of rows in file"] = (100 * cleaning["total"] / cleaning.loc["rows in file", "total"]).round(2)
cleaning.to_csv(f"{OUTPUTS}/cleaning.csv")
display(cleaning.style.format("{:,.0f}", subset=cleaning.columns[:-1]).format("{:.2f}", subset=["% of rows in file"]))
nulls = pd.DataFrame({s["month"]: s["nulls_filled"] for s in stats}).sum(axis=1)
print("nulls filled after cleaning (all months):", nulls.to_dict())
print("per-month counts saved to outputs/cleaning_per_month.csv")

## Part 2 · Combining the partial results, and checking them

**Sharding:** one shard per month. A month is a natural unit: the files are already split that
way, each shard keeps its own time order, and the spill-over rule needs to know the month.
The 33 files are of similar size (see the footer table), but the 2026 test months are not
sampled, so they write more rows and take longer. This load imbalance is measured in Part 5.

**Combining:** each worker returns *counts* (trips and cash trips per zone, training rows only).
The main process adds the counts over the shards in a fixed shard order, then divides.

**Check 1:** the parallel run must give exactly the same cleaned shards as a sequential run
(checked on the first `BENCH_SHARDS` files, which Part 5 also uses).
**Check 2:** the merged zone rates must equal rates computed directly from all training rows at once.

In [ ]:
bench_shards = list(range(min(BENCH_SHARDS, len(jobs))))
seq_paths = [f"{OUT}/clean_seq/{m}_{r}.parquet" for m, r in zip(months, roles)]
seq_stats = run_tasks(pw.clean_month, clean_args(bench_shards, seq_paths), 1)
check1 = [stats[i]["md5"] == b["md5"] for i, b in zip(bench_shards, seq_stats)]
print("check 1 - parallel shards identical to sequential shards:", all(check1), check1)

def zone_rates(stats, prefix):
    """Merge partial counts (fixed shard order), then a smoothed cash rate per zone."""
    train = [s for s in stats if s["role"] == "train"]
    n    = pd.concat([s[f"{prefix}_n"] for s in train], axis=1).fillna(0).sum(axis=1)
    cash = pd.concat([s[f"{prefix}_cash"] for s in train], axis=1).fillna(0).sum(axis=1)
    prior = cash.sum() / n.sum()
    return (cash + SMOOTHING * prior) / (n + SMOOTHING), prior

pu_rate, prior = zone_rates(stats, "pu")
do_rate, _ = zone_rates(stats, "do")

# the same rates computed in one go from all training rows (no sharding)
allrows = pd.concat([pd.read_parquet(p, columns=["PULocationID", "split", "cash"])
                     for p, r in zip(clean_paths, roles) if r == "train"])
tr = allrows[allrows.split == pw.TRAIN]
direct = (tr.groupby("PULocationID").cash.sum() + SMOOTHING * tr.cash.mean()) / \
         (tr.groupby("PULocationID").size() + SMOOTHING)
check2 = np.allclose(direct.sort_index().values, pu_rate.sort_index().values, rtol=0, atol=1e-12)
print("check 2 - merged zone rates equal the direct calculation:", check2)
print(f"overall training cash rate {prior:.2%}; {len(pu_rate)} pickup zones, {len(do_rate)} drop-off zones")

## Part 3 · Feature extraction (task parallelism #1)

The 27 features fall into four groups that need **different input columns** and do not depend
on each other. Each group is a separate task, so the four run at the same time:

| Task | Features |
|---|---|
| `trip` | distance, duration, speed, fare, fare per mile, extra, tolls, passengers |
| `time` | hour (as sine/cosine), weekend, night, rush hour |
| `flags` | airport / congestion / CBD fee, same zone, store-and-forward, rate code (one-hot), vendor |
| `zones` | pickup and drop-off zone cash rate (from Part 2, training rows only) |

The groups are joined side by side in the **fixed order** `pw.GROUP_ORDER`, so column *i*
always means the same feature (Part 6 shows what goes wrong otherwise).

In [ ]:
groups = pw.GROUP_ORDER
feat_paths = [f"{OUT}/features/{g}.npy" for g in groups]
zone_tables = (pu_rate, do_rate, prior)

def feature_args():
    return [groups, [clean_paths] * len(groups), feat_paths, [zone_tables] * len(groups)]

feats, feat_s = timed(run_tasks, pw.build_feature_group, feature_args(), len(groups))
display(pd.DataFrame(feats)[["group", "rows", "seconds", "pid", "md5"]])
print(f"4 feature tasks in {feat_s:.1f}s wall time; slowest task {max(f['seconds'] for f in feats):.1f}s")

columns, X = pw.assemble_features(dict(zip(groups, feat_paths)))
labels = pd.concat([pd.read_parquet(p, columns=["split", "cash"]).assign(month=m)
                    for p, m in zip(clean_paths, months)], ignore_index=True)
assert len(labels) == len(X)
sizes = {}
for name, code_ in (("train", pw.TRAIN), ("val", pw.VAL), ("test", pw.TEST)):
    m = (labels.split == code_).to_numpy()
    np.save(f"{OUT}/prepared/{name}_X.npy", X[m])
    np.save(f"{OUT}/prepared/{name}_y.npy", labels.cash.to_numpy(np.int8)[m])
    sizes[name] = {"trips": int(m.sum()), "cash_share": float(labels.cash[m].mean())}
np.save(f"{OUT}/prepared/test_month.npy", labels.month[labels.split == pw.TEST].to_numpy().astype("U7"))
del X
print(f"{len(columns)} features:", ", ".join(columns))
pd.DataFrame(sizes).T

## Part 4 · Hyperparameter search (task parallelism #2)

Each of the six model settings is a complete, independent training job (it reads the same
training data and writes nothing the others need), so they run as six separate tasks.
Every job is trained on January 2024 – September 2025 and scored on the **validation window**
(October – December 2025). The best setting is chosen on validation F1 for the cash class
(cash is only about 12–16% of trips, so accuracy alone would reward always saying "card").
The 2026 test months are not touched until Part 7.

In [ ]:
names, kinds, params = map(list, zip(*GRID))
k = len(GRID)
prep_dir, models_dir = f"{OUT}/prepared", f"{OUT}/models"
runs, search_s = timed(run_tasks, pw.train_one_setting,
                       [names, kinds, params, [prep_dir] * k, [models_dir] * k], min(MAX_WORKERS, k))
search = pd.DataFrame(runs)
search.to_csv(f"{OUTPUTS}/model_search.csv", index=False)
display(search[["name", "val_auc", "val_f1_at_0.5", "val_f1_tuned", "threshold", "seconds", "pid"]]
        .style.format(precision=4))
best = search.sort_values(["val_f1_tuned", "val_auc"], ascending=False).iloc[0]
print(f"wall time {search_s:.1f}s; sum of task times {search.seconds.sum():.1f}s; "
      f"slowest task {search.seconds.max():.1f}s ({search.loc[search.seconds.idxmax(), 'name']})")
print(f"chosen on validation: {best['name']} (F1 cash {best['val_f1_tuned']:.4f} at threshold {best['threshold']:.2f})")

## Part 5 · Sequential vs parallel

The same workload is run with 1 worker (a plain loop in this process) and with more workers.
Each stage gets **one untimed warm-up run** first (it fills the operating system's file cache and
loads the libraries, so the first timed run is not unfairly slow). Then every worker count gets
**3 timed runs** (`REPEATS`), and we report the mean. Every run's output is fingerprinted and
compared with the 1-worker output.

Stages benchmarked:
* cleaning (data parallelism), on the first 8 monthly files (`BENCH_SHARDS`), with 1, 2, 4 and 8 workers
* feature extraction (task parallelism), all four groups on the full data, with 1, 2 and 4 workers
* model search (task parallelism), all six settings trained on the first `BENCH_ROWS` training rows
  (so the benchmark stays short on the shared server), with 1, 2, 3 and 6 workers

In [ ]:
def benchmark(stage, fn, make_args, counts, fingerprint):
    rows, reference = [], None
    run_tasks(fn, make_args(), max(counts))                  # warm-up, not timed
    for n in counts:
        for rep in range(1, REPEATS + 1):
            result, sec = timed(run_tasks, fn, make_args(), n)
            fp = fingerprint(result)
            reference = fp if reference is None else reference
            rows.append({"stage": stage, "workers": n, "repeat": rep, "seconds": sec,
                         "same_output_as_1_worker": fp == reference})
    return rows

bench = []
bench += benchmark("cleaning (data parallel)", pw.clean_month, lambda: clean_args(bench_shards),
                   CLEAN_WORKER_COUNTS, lambda res: tuple(r["md5"] for r in res))
bench += benchmark("feature extraction (task parallel)", pw.build_feature_group, feature_args,
                   FEATURE_WORKER_COUNTS, lambda res: tuple(r["md5"] for r in res))
search_args = lambda: [names, kinds, params, [prep_dir] * k, [None] * k, [BENCH_ROWS] * k]
bench += benchmark("model search (task parallel)", pw.train_one_setting, search_args,
                   SEARCH_WORKER_COUNTS, lambda res: tuple(r["val_auc"] for r in res))
bench = pd.DataFrame(bench)
bench.to_csv(f"{OUTPUTS}/benchmark_runs.csv", index=False)

summary_rows = []
for stage, g in bench.groupby("stage", sort=False):
    t1 = g[g.workers == 1].seconds.mean()
    for n, h in g.groupby("workers"):
        S = t1 / h.seconds.mean()
        summary_rows.append({"stage": stage, "workers": n, "mean_s": h.seconds.mean(),
                             "std_s": h.seconds.std(), "speedup": S, "efficiency": S / n,
                             # Amdahl: S = 1 / ((1-p) + p/n)  ->  p = (1 - 1/S) / (1 - 1/n)
                             "amdahl_p": (1 - 1 / S) / (1 - 1 / n) if n > 1 else np.nan,
                             "outputs_match": bool(h.same_output_as_1_worker.all())})
speed = pd.DataFrame(summary_rows)
speed.to_csv(f"{OUTPUTS}/speedup.csv", index=False)
print("all parallel outputs identical to the 1-worker outputs:", bool(bench.same_output_as_1_worker.all()))
speed.style.format({"mean_s": "{:.2f}", "std_s": "{:.2f}", "speedup": "{:.2f}x",
                    "efficiency": "{:.0%}", "amdahl_p": "{:.2f}"})

### Speed-up chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ALL_COUNTS = sorted(set(CLEAN_WORKER_COUNTS + FEATURE_WORKER_COUNTS + SEARCH_WORKER_COUNTS))
top = max(ALL_COUNTS)
ax.plot([1, top], [1, top], "--", color="grey", label="ideal (speed-up = workers)")
for stage, g in speed.groupby("stage", sort=False):
    ax.plot(g.workers, g.speedup, "o-", lw=2, label=stage)
    for n, s in zip(g.workers, g.speedup):
        ax.annotate(f"{s:.1f}x", (n, s), textcoords="offset points", xytext=(6, -12), fontsize=9)
ax.set_xlabel("worker processes"); ax.set_ylabel(f"speed-up (mean of {REPEATS} runs)")
ax.set_xticks(ALL_COUNTS)
ax.set_title("Speed-up against number of workers (Rockhill)")
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(f"{OUTPUTS}/speedup.png", dpi=200); plt.show()

### Why the speed-up is below ideal: Amdahl's law and measured overheads
Amdahl's law says that if a fraction *p* of the work can run in parallel, then
*S(n) = 1 / ((1 − p) + p / n)*. Re-arranging gives the estimate of *p* in the table above.
The cell below measures the overheads that make up the "serial" part.

In [ ]:
# 1. process start-up: an empty pool with n workers doing nothing
startup = {}
for n in [c for c in ALL_COUNTS if c > 1]:
    _, startup[n] = timed(run_tasks, pw.noop, [list(range(n))], n)

# 2. load imbalance: with as many workers as tasks, wall time >= the slowest task
def imbalance(task_seconds):
    return max(task_seconds) / (sum(task_seconds) / len(task_seconds))

clean_tasks  = [s["seconds"] for s in seq_stats]
feat_tasks   = [f["seconds"] for f in feats]
search_tasks = list(search.seconds)

# 3. data transfer: size of what each worker sends back (pickled)
import pickle
transfer_kb = sum(len(pickle.dumps(s)) for s in stats) / 1024

overheads = pd.DataFrame([
    {"overhead": f"pool start-up, {n} workers", "value": f"{s:.2f} s"} for n, s in startup.items()] + [
    {"overhead": "cleaning: slowest shard / average shard", "value": f"{imbalance(clean_tasks):.2f}x"},
    {"overhead": "features: slowest task / average task", "value": f"{imbalance(feat_tasks):.2f}x"},
    {"overhead": "search: slowest model / average model", "value": f"{imbalance(search_tasks):.2f}x"},
    {"overhead": "search: slowest model share of all model time", "value": f"{max(search_tasks) / sum(search_tasks):.0%}"},
    {"overhead": "cleaning: partial results sent back to main process", "value": f"{transfer_kb:.0f} KB"},
])
overheads.to_csv(f"{OUTPUTS}/overheads.csv", index=False)
display(overheads)

for stage, g in speed.groupby("stage", sort=False):
    last = g.iloc[-1]
    msg = (f"{stage}: {last.speedup:.2f}x on {int(last.workers)} workers "
           f"(efficiency {last.efficiency:.0%}), Amdahl estimate p = {last.amdahl_p:.2f}")
    if last.amdahl_p > 0:
        msg += f" -> at most {1 / (1 - min(last.amdahl_p, 0.999)):.1f}x with unlimited workers"
    else:
        msg += " -> overheads cost more than the parallel work saves at this data size"
    print(msg)

**Reading the numbers.**
* **Load imbalance** is the main limit for the task-parallel stages. With one task per worker,
  the stage cannot finish before its slowest task. In the search, the large boosted-tree models
  take far longer than the logistic regressions, so extra workers sit idle at the end.
* **I/O.** Cleaning is mostly reading and decompressing Parquet. All workers share the same disk
  and memory bandwidth, so this part does not scale linearly.
* **Process start-up and data transfer.** Starting a pool costs a fraction of a second, and
  workers send back only small count tables, so these overheads are small here.
* **Shared server.** Other groups use Rockhill at the same time (see the load average in
  Part 0), which adds noise. That is why we report the mean and standard deviation of repeated runs.

## Part 6 · Non-determinism: two sources, shown and fixed

For each source we run the **same code on the same data twice without the fix** (outputs differ),
then **twice with the fix** (outputs identical).

### Source 1 · Unseeded randomness in sampling
`clean_month` takes a random 5% sample of each training month. Without a seed, each run
picks different rows, so the zone cash rates and every model trained afterwards change.
**Fix:** a fixed seed per shard (`SEED + shard index`). The seed belongs to the shard, not to a
worker, so the result does not depend on which worker cleans which month.

In [ ]:
def demo_clean(seed, tag):
    s = pw.clean_month(paths[0], months[0], f"{OUT}/demo/{tag}.parquet", "train", SAMPLE, seed)
    zone = 132 if 132 in s["pu_n"].index else int(s["pu_n"].idxmax())   # 132 = JFK airport
    return {"source": f"unseeded sampling ({months[0]})", "run": tag, "output fingerprint": s["md5"],
            "rows kept": s["rows_kept"],
            f"cash rate, pickup zone {zone}": f"{s['pu_cash'][zone] / s['pu_n'][zone]:.6f}"}

nd1 = pd.DataFrame([demo_clean(None, "no seed, run 1"), demo_clean(None, "no seed, run 2"),
                    demo_clean(SEED, "seed 4207, run 1"), demo_clean(SEED, "seed 4207, run 2")])
nd1

### Source 2 · Set / hash ordering when joining the feature groups
A first version collected the finished feature groups with `set(...)` and joined them in that
order. Python randomises string hashing **in every new process**, so the order of a set of
strings changes from run to run. The columns of X then come out in a different order, and a
saved model or its feature weights silently refer to the wrong features.
We run the join in **separate Python processes**, as separate runs of the program would.
**Fix:** always join in the fixed list `pw.GROUP_ORDER`.

In [ ]:
group_files = json.dumps(dict(zip(groups, feat_paths)))
snippet = ("import json, sys, payment_workers as pw\n"
           "cols, X = pw.assemble_features(json.loads(sys.argv[1]), naive=sys.argv[2] == '1', rows=100000)\n"
           "order = list(dict.fromkeys(g for c in cols for g in pw.GROUP_ORDER if c in pw.GROUP_COLUMNS[g]))\n"
           "print(json.dumps({'order': ' | '.join(order), 'md5': pw.digest(X[:100000])}))")
clean_env = {k: v for k, v in os.environ.items() if k != "PYTHONHASHSEED"}   # default: random per process

def demo_join(naive, run):
    out = subprocess.run([sys.executable, "-c", snippet, group_files, "1" if naive else "0"],
                         capture_output=True, text=True, env=clean_env, cwd=os.getcwd(), check=True)
    r = json.loads(out.stdout)
    return {"source": "set ordering of feature groups",
            "run": f"{'set() order' if naive else 'fixed GROUP_ORDER'}, run {run}",
            "column-group order": r["order"], "output fingerprint": r["md5"]}

nd2 = pd.DataFrame([demo_join(True, i) for i in (1, 2, 3, 4)] + [demo_join(False, i) for i in (1, 2, 3, 4)])
pd.concat([nd1, nd2]).to_csv(f"{OUTPUTS}/nondeterminism.csv", index=False)
nd2

### Other sources we prevented, and the limits of the fixes
* **Completion order:** results are collected with `pool.map`, which returns them in input
  (shard/task) order, never with `as_completed`. Shards are also combined in a fixed order.
* **Thread-count dependent maths:** every process is pinned to one maths thread, so results do
  not depend on how many cores a library decides to use.
* **Row order inside a shard:** rows are sorted with a stable sort before sampling.

**Limits.** The fixes make the result identical for the same data, code and library versions.
A different version of numpy, pandas or scikit-learn (see Part 0) can change the random stream
or the floating-point rounding, so `requirements.txt` records the versions we used. Results are
also only reproducible for the same `SAMPLE`, `SEED` and month lists.

## Part 7 · Final test on all of 2026 (never seen during training or model choice)

In [ ]:
import pickle
Xt = np.load(f"{prep_dir}/test_X.npy"); yt = np.load(f"{prep_dir}/test_y.npy")
with open(f"{models_dir}/{best['name']}.pkl", "rb") as fh:
    model = pickle.load(fh)
p = model.predict_proba(Xt)[:, 1]
pred = (p >= best["threshold"]).astype(int)
tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
test = {"model": best["name"], "threshold": float(best["threshold"]), "test_trips": int(len(yt)),
        "accuracy": accuracy_score(yt, pred), "precision_cash": precision_score(yt, pred),
        "recall_cash": recall_score(yt, pred), "f1_cash": f1_score(yt, pred),
        "f1_macro": f1_score(yt, pred, average="macro"), "roc_auc": roc_auc_score(yt, p),
        "baseline_always_card_accuracy": float(1 - yt.mean()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}
display(pd.Series(test, name="2026 (Jan-Sep)").to_frame())

from sklearn.metrics import ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(np.array([[tn, fp], [fn, tp]]), display_labels=["card", "cash"])
disp.plot(values_format=",", cmap="Blues")
plt.title(f"Test 2026, {best['name']} (threshold {best['threshold']:.2f})")
plt.tight_layout(); plt.savefig(f"{OUTPUTS}/confusion_matrix.png", dpi=200); plt.show()

### Does it hold up month by month?
The model was trained on 2024–2025 only. If riders' habits drift (for example, fewer people
paying cash), the scores should slowly fall during 2026. We also show the cash share for
every month in all three periods.

In [ ]:
tm = np.load(f"{prep_dir}/test_month.npy")
by_month = pd.DataFrame([{
    "month": mo, "trips": int((tm == mo).sum()), "cash_share": float(yt[tm == mo].mean()),
    "roc_auc": roc_auc_score(yt[tm == mo], p[tm == mo]),
    "f1_cash": f1_score(yt[tm == mo], pred[tm == mo]),
    "recall_cash": recall_score(yt[tm == mo], pred[tm == mo])} for mo in sorted(set(tm))])
by_month.to_csv(f"{OUTPUTS}/test_by_month.csv", index=False)
display(by_month.style.format({"trips": "{:,}", "cash_share": "{:.1%}", "roc_auc": "{:.3f}",
                               "f1_cash": "{:.3f}", "recall_cash": "{:.1%}"}))

share = pd.DataFrame([{"month": s["month"], "role": s["role"], "cash_share": s["cash_share"]} for s in stats])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
for role, colour in (("train", "tab:blue"), ("val", "tab:orange"), ("test", "tab:green")):
    r = share[share.role == role]
    a1.plot(pd.to_datetime(r.month), 100 * r.cash_share, "o-", color=colour, label=role)
a1.set_ylabel("cash trips (% of card + cash)"); a1.set_title("Cash share by month"); a1.legend(); a1.grid(alpha=0.3)
a2.plot(pd.to_datetime(by_month.month), by_month.roc_auc, "o-", label="ROC AUC")
a2.plot(pd.to_datetime(by_month.month), by_month.f1_cash, "s-", label="F1 (cash)")
a2.set_title("Test scores by 2026 month"); a2.legend(); a2.grid(alpha=0.3)
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(f"{OUTPUTS}/by_month.png", dpi=200); plt.show()

### Which features point to cash? (weights of the best logistic regression, standardised features)

In [ ]:
lr_name = search[search.kind == "logreg"].sort_values("val_f1_tuned", ascending=False).iloc[0]["name"]
with open(f"{models_dir}/{lr_name}.pkl", "rb") as fh:
    weights = pd.Series(pickle.load(fh)[-1].coef_[0], index=columns).sort_values()
weights.to_csv(f"{OUTPUTS}/logreg_weights.csv", header=["weight"])
print(f"model: {lr_name}   (positive = towards cash)")
pd.concat([weights.head(5), weights.tail(5)]).to_frame("weight").style.format("{:+.2f}")

## Part 8 · Summary (every number used on the poster comes from here)

In [ ]:
summary = {
    "environment": env, "config": {"train": TRAIN_MONTHS, "val": VAL_MONTHS, "test": TEST_MONTHS,
                                   "sample": SAMPLE, "test_sample": TEST_SAMPLE,
                                   "seed": SEED, "repeats": REPEATS, "bench_rows": BENCH_ROWS},
    "cleaning": cleaning["total"].to_dict(), "sizes": sizes,
    "checks": {"parallel_equals_sequential_cleaning": all(check1),
               "merged_zone_rates_equal_direct": bool(check2),
               "all_benchmark_outputs_match": bool(bench.same_output_as_1_worker.all())},
    "speedup": speed.to_dict(orient="records"), "overheads": overheads.to_dict(orient="records"),
    "best_model": best["name"], "search": search.drop(columns=["params"]).to_dict(orient="records"),
    "test": test, "test_by_month": by_month.to_dict(orient="records"),
    "cash_share_by_month": share.to_dict(orient="records"),
    "top_weights": pd.concat([weights.head(5), weights.tail(5)]).to_dict(),
    "nondeterminism": pd.concat([nd1, nd2]).astype(str).to_dict(orient="records"),
    "notebook_minutes": (time.perf_counter() - NOTEBOOK_START) / 60,
}
with open(f"{OUTPUTS}/summary.json", "w") as fh:
    json.dump(summary, fh, indent=2, default=float)
print(f"notebook run time: {summary['notebook_minutes']:.1f} minutes on {env['host']} "
      f"({env['usable_cores']} usable cores)")
print("checks:", summary["checks"])
print(f"test: AUC {test['roc_auc']:.3f}, F1 cash {test['f1_cash']:.3f}, recall cash {test['recall_cash']:.1%} "
      f"on {test['test_trips']:,} trips")